In [ ]:
import pandas as pd
import requests
import re

from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

url = "https://opendata.dc.gov/api/feed/dcat-us/1.1.json"

response = requests.get(url)
response.raise_for_status()

catalog = response.json()

print(catalog.keys())

dict_keys(['@context', '@type', 'conformsTo', 'describedBy', 'dataset'])


In [ ]:
datasets = catalog["dataset"]

print("Number of datasets:", len(datasets))

Number of datasets: 1869


In [ ]:
for dataset in datasets[:5]:
    print(dataset.get("title"))

Parking Violations Issued in April 2014
Parking Violations Issued in February 2009
Interagency Data Team
Data Policy
Open Data Handbook


In [ ]:
identifiers = [
    d.get("identifier")
    for d in datasets
    if d.get("identifier") is not None
]

print("Dataset entries:", len(datasets))
print("Unique identifiers:", len(set(identifiers)))

Dataset entries: 1869
Unique identifiers: 1869


In [ ]:
# Find arcgis dataset layers
arcgis_layers = defaultdict(set)

for dataset in datasets:

    title = dataset.get("title", "Unknown dataset")

    for dist in dataset.get("distribution", []):

        url = (
            dist.get("accessURL")
            or dist.get("downloadURL")
        )

        if not url:
            continue

        clean_url = url.split("?")[0].rstrip("/")

        match = re.search(
            r"(https?://.+?/(?:MapServer|FeatureServer)/\d+)$",
            clean_url,
            re.IGNORECASE
        )

        if match:

            layer_url = match.group(1)

            arcgis_layers[layer_url].add(title)


print(
    "Unique ArcGIS layer URLs:",
    len(arcgis_layers)
)

Unique ArcGIS layer URLs: 1443


In [ ]:
# Records for each layer
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

def get_record_count(layer_url):

    try:

        response = requests.get(
            f"{layer_url}/query",
            params={
                "where": "1=1",
                "returnCountOnly": "true",
                "f": "json"
            },
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

        return data.get("count")

    except Exception:
        return None


record_results = []


with ThreadPoolExecutor(max_workers=15) as executor:

    futures = {
        executor.submit(
            get_record_count,
            layer_url
        ): layer_url

        for layer_url in arcgis_layers
    }

    for future in as_completed(futures):

        layer_url = futures[future]
        count = future.result()

        if count is not None:

            titles = arcgis_layers[layer_url]

            record_results.append({
                "Dataset": " | ".join(
                    sorted(titles)
                ),
                "Records": count,
                "Layer URL": layer_url
            })


dc_counts = pd.DataFrame(
    record_results
)

print(
    "Successfully counted:",
    len(dc_counts)
)

Successfully counted: 1431


In [ ]:
record_counts = dc_counts["Records"]

print("ArcGIS layers analyzed:", len(record_counts))
print(f"Total records: {record_counts.sum():,}")
print(f"Average records: {record_counts.mean():,.0f}")
print(f"Median records: {record_counts.median():,.0f}")
print(f"Minimum records: {record_counts.min():,}")
print(f"Maximum records: {record_counts.max():,}")

ArcGIS layers analyzed: 1431
Total records: 96,082,855
Average records: 67,144
Median records: 13,156
Minimum records: 0
Maximum records: 2,053,654


In [ ]:
top_dc = (
    dc_counts
    .sort_values(
        "Records",
        ascending=False
    )
    .reset_index(drop=True)
)

top_dc.head(4)

,Dataset,Records,Layer URL
0,Street Tree Archive,2053654,https://maps2.dcgis.dc.gov/DCGIS/rest/services...
1,DC Tree Structure and Benefits,1985917,https://maps2.dcgis.dc.gov/dcgis/rest/services...
2,DC Trees,1985917,https://maps2.dcgis.dc.gov/dcgis/rest/services...
3,Payments from PASS,1568923,https://maps2.dcgis.dc.gov/dcgis/rest/services...


In [ ]:
pd.set_option("display.max_colwidth", None)

top_dc.head(10)[
    ["Dataset", "Records", "Layer URL"]
]

,Dataset,Records,Layer URL
0,Street Tree Archive,2053654,https://maps2.dcgis.dc.gov/DCGIS/rest/services/DCGIS_DATA/Urban_Tree_Canopy/FeatureServer/13
1,DC Tree Structure and Benefits,1985917,https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Urban_Tree_Canopy/MapServer/12
2,DC Trees,1985917,https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Urban_Tree_Canopy/MapServer/11
3,Payments from PASS,1568923,https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Government_Operations/MapServer/17
4,Cityworks Service Requests,1503280,https://maps2.dcgis.dc.gov/dcgis/rest/services/DDOT/Cityworks/MapServer/1
5,Cityworks Workorders,1303341,https://maps2.dcgis.dc.gov/dcgis/rest/services/DDOT/Cityworks/MapServer/0
6,DC Estimated Trees,1231140,https://services.arcgis.com/neT9SoYxizqTHZPH/arcgis/rest/services/DC_Estimated_Trees/FeatureServer/0
7,Occupancy Permits (via DDOT TOPs),974674,https://maps2.dcgis.dc.gov/dcgis/rest/services/DDOT/TOPS/MapServer/1
8,Crash Details Table,901052,https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Public_Safety_WebMercator/MapServer/25
9,DMV Vehicle Inspections,792279,https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Transportation_DMV_WebMercator/MapServer/1


In [ ]:
def get_layer_metadata(layer_url):

    try:
        response = requests.get(
            layer_url,
            params={"f": "json"},
            timeout=30
        )

        response.raise_for_status()
        data = response.json()

        return {
            "name": data.get("name"),
            "geometryType": data.get("geometryType"),
            "fields": len(data.get("fields", [])),
            "objectIdField": data.get("objectIdField"),
            "maxRecordCount": data.get("maxRecordCount")
        }

    except Exception as e:
        return {"error": str(e)}


metadata_rows = []

for _, row in top_dc.head(10).iterrows():

    meta = get_layer_metadata(
        row["Layer URL"]
    )

    metadata_rows.append({
        "Dataset": row["Dataset"],
        "Records": row["Records"],
        "Layer URL": row["Layer URL"],
        **meta
    })


metadata_df = pd.DataFrame(metadata_rows)

metadata_df

,Dataset,Records,Layer URL,name,geometryType,fields,objectIdField,maxRecordCount
0,Street Tree Archive,2053654,https://maps2.dcgis.dc.gov/DCGIS/rest/services/DCGIS_DATA/Urban_Tree_Canopy/FeatureServer/13,Street Tree Archive,None,13,OBJECTID,2000
1,DC Tree Structure and Benefits,1985917,https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Urban_Tree_Canopy/MapServer/12,DC Tree Structure and Benefits,None,40,None,2000
2,DC Trees,1985917,https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Urban_Tree_Canopy/MapServer/11,DC Trees,esriGeometryPoint,22,None,2000
3,Payments from PASS,1568923,https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Government_Operations/MapServer/17,PASS Payments,None,22,None,1000
4,Cityworks Service Requests,1503280,https://maps2.dcgis.dc.gov/dcgis/rest/services/DDOT/Cityworks/MapServer/1,CityworksRequest,esriGeometryPoint,55,None,2000
5,Cityworks Workorders,1303341,https://maps2.dcgis.dc.gov/dcgis/rest/services/DDOT/Cityworks/MapServer/0,CityworksWorkorder,esriGeometryPoint,60,None,2000
6,DC Estimated Trees,1231140,https://services.arcgis.com/neT9SoYxizqTHZPH/arcgis/rest/services/DC_Estimated_Trees/FeatureServer/0,DC_estimated_trees,esriGeometryPoint,4,OBJECTID,2000
7,Occupancy Permits (via DDOT TOPs),974674,https://maps2.dcgis.dc.gov/dcgis/rest/services/DDOT/TOPS/MapServer/1,Occupancy Permit,esriGeometryPoint,87,None,2000
8,Crash Details Table,901052,https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Public_Safety_WebMercator/MapServer/25,Crash Details,None,15,None,1000
9,DMV Vehicle Inspections,792279,https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Transportation_DMV_WebMercator/MapServer/1,DMV Vehicle Inspections,None,10,None,2000
